## State

---

> **In one line.** Model an object as a **finite state machine** $M$ that carries a current state $q$ and hands every incoming input $\sigma$ to it. The transition $\delta(q, \sigma) = q'$ is computed *inside the state object* — each state knows what comes next — so the context never branches on `if/elif`.

### 1. The machine

The pattern formalizes an object's behaviour as a **finite state machine**, the five-tuple

$$M = (Q,\ \Sigma,\ \delta,\ q_0,\ F).$$

Here $Q$ is the finite **set of states** — for a vending machine, $Q = \{\text{Idle}, \text{HasMoney}, \text{Dispensing}, \text{OutOfStock}\}$. At any moment the machine sits in exactly one **current state** $q \in Q$, which in code is a *state object* holding precisely the behaviour valid while the machine is there. The machine begins in the **initial state** $q_0$ (say, Idle), and $F \subseteq Q$ collects the **terminal/accepting states** — those from which no further transition fires (e.g. OutOfStock).

Inputs are drawn from the **alphabet** $\Sigma$, the finite set of events the machine reacts to — e.g. $\Sigma = \{\text{insert\_money}, \text{select\_product}, \text{cancel}\}$. A single event is a symbol $\sigma \in \Sigma$, and each $\sigma$ is a *potential* trigger for change.

### 2. The transition function

The heart of $M$ is the **transition function** $\delta$. Given where the machine is ($q$) and what just arrived ($\sigma$), it names where the machine goes next ($q'$):

$$\boxed{\,\delta : Q \times \Sigma \longrightarrow Q\,}, \qquad \delta(q,\ \sigma) = q'.$$

Reading an input therefore advances the machine by one step along the chain

$$\underbrace{q}_{\text{current state}} \;\xrightarrow{\;\sigma\;}\; \underbrace{\delta(q, \sigma)}_{\;=\,q'\,\in\,Q} \;\xrightarrow{\;\sigma'\;}\; \cdots$$

so a run is just the repeated application of $\delta$ to a stream of symbols out of $\Sigma$, starting from $q_0$.

### 3. Context delegation

The object-oriented realization splits this cleanly. The **context** $M$ stores the current state as `self._state = q` and, for every event $\sigma$, does nothing but **delegate** — it forwards the call straight to the state object rather than inspecting $q$ itself:

$$\text{event } \sigma \;\xrightarrow{\;\text{delegate}\;}\; q.\sigma() \;\xrightarrow{\;\delta\;}\; q' .$$

Crucially, $\delta$ lives *inside* each state: the state object computes $q'$ and installs it by calling `context.set_state(q')`. The context owns the data ($q$); the states own the logic ($\delta$).

### 4. Key conditions

1. **No `if/elif` chains.** The context holds `self._state = q` and delegates every method call to it. Each test like `if self._state == "idle"` is replaced by a plain method call on the state object.
   $$M.\sigma \;=\; \big(\,q.\sigma\,\big), \qquad q = \texttt{self.\_state}.$$
2. **Self-directed transitions.** Each state $q$ knows its own outgoing edges $\delta(q, \sigma) = q'$ and performs the move itself, via `context.set_state(q')`.
3. **Invalid transitions.** When $\delta(q, \sigma)$ is **undefined**, the state raises a clear error inherited from a base-class default method — partial functions are made explicit, not silently ignored.

&nbsp;

> 🚦 A traffic light. $Q = \{\text{Red}, \text{Green}, \text{Yellow}\}$, with $\delta(\text{Red}, \text{tick}) = \text{Green}$. Each state object knows what comes next; the light merely holds the current state and delegates — zero `if colour == "red"` logic.

### Exercise 11 — Vending Machine

---

**Scenario:** $Q = \{\text{Idle, HasMoney, Dispensing, OutOfStock}\}$. $\Sigma = \{\text{insert\_money, select\_product, cancel}\}$. Each $(q, \sigma)$ pair has a defined behaviour and transition $\delta(q, \sigma) = q'$.

**Your task:** Implement each state as a class. The machine context holds $q$ and delegates all method calls to it.

```python
vm = VendingMachine()    # q_0 = Idle
vm.insert_money()        # δ(Idle, insert) = HasMoney
vm.select_product()      # δ(HasMoney, select) = Dispensing → Idle
vm.select_product()      # δ(Idle, select) → error: no money
```

**Hints**

- The machine holds `self._state = q`. Every action delegates: `self._state.insert_money()`. Each state calls `context.set_state(q')` to implement $\delta$.
- Draw the FSM first — list every $(q, \sigma) \rightarrow q'$ transition explicitly. This becomes your implementation plan.

In [ ]:
# --------------------------------
# Context M — holds current state q and delegates every input to it (no if/elif)

class VendingMachine:
    def __init__(self):
        self._state = IdleState(self)        # q_0 = Idle

    def set_state(self, state):              # used by states to implement δ(q, σ) = q'
        self._state = state

    def insert_money(self):                  # delegate σ = insert_money to q
        self._state.insert_money()

    def select_product(self):                # delegate σ = select_product to q
        self._state.select_product()

    def cancel(self):                        # delegate σ = cancel to q
        self._state.cancel()

# --------------------------------
# Base state — default behaviour: an undefined δ(q, σ) is an error

class State:
    def __init__(self, machine):
        self._machine = machine              # reference to context M

    def insert_money(self):
        print(f"  [{type(self).__name__}] cannot insert money here")
    def select_product(self):
        print(f"  [{type(self).__name__}] cannot select a product here")
    def cancel(self):
        print(f"  [{type(self).__name__}] nothing to cancel")

# --------------------------------
# Each q in Q is a class. Each defined method implements δ(q, σ) = q' via set_state.

class IdleState(State):                      # q = Idle
    def insert_money(self):
        # δ(Idle, insert_money) = HasMoney
        ...

class HasMoneyState(State):                  # q = HasMoney
    def select_product(self):
        # δ(HasMoney, select_product) = Dispensing, then dispense -> Idle
        ...
    def cancel(self):
        # δ(HasMoney, cancel) = Idle (refund)
        ...

class DispensingState(State):                # q = Dispensing
    ...                                      # transient state; inherits base error defaults

# --------------------------------
vm = VendingMachine()        # q_0 = Idle
vm.insert_money()            # δ(Idle, insert) = HasMoney
vm.select_product()          # δ(HasMoney, select) = Dispensing -> Idle
vm.select_product()          # δ(Idle, select) -> error: no money

### Exercise 12 — Order Lifecycle

---

**Scenario:** $Q = \{\text{Pending, Confirmed, Shipped, Delivered, Cancelled}\}$. Not all $\delta(q, \sigma)$ are defined — a Delivered order cannot transition to Cancelled.

**Your task:** Model each state as a class. Invalid $\delta(q, \sigma)$ raises `InvalidTransitionError` from a base class default.

```python
order = Order()          # q_0 = Pending
order.confirm()          # δ(Pending, confirm) = Confirmed
order.ship()             # δ(Confirmed, ship) = Shipped
order.deliver()          # δ(Shipped, deliver) = Delivered
order.cancel()           # δ(Delivered, cancel) undefined -> InvalidTransitionError
```

**Hints**

- Define $\delta$ explicitly as a table before coding. States that support a transition implement that method; others inherit the base class's error-raising default.
- The base `OrderState` default for every $\sigma$ raises `InvalidTransitionError`. Each concrete state overrides only the transitions it allows, calling `context.set_state(q')`.

In [ ]:
class InvalidTransitionError(Exception):
    pass

# --------------------------------
# Context M — holds q, delegates each input σ to the current state

class Order:
    def __init__(self):
        self._state = PendingState(self)     # q_0 = Pending

    def set_state(self, state):              # implements δ(q, σ) = q'
        self._state = state
        print(f"  -> now {type(state).__name__}")

    def confirm(self):  self._state.confirm()
    def ship(self):     self._state.ship()
    def deliver(self):  self._state.deliver()
    def cancel(self):   self._state.cancel()

# --------------------------------
# Base state — default for every σ: δ(q, σ) is undefined -> raise

class OrderState:
    def __init__(self, machine):
        self._machine = machine

    def _invalid(self, action):
        raise InvalidTransitionError(
            f"cannot '{action}' from {type(self).__name__}"
        )

    def confirm(self):  self._invalid("confirm")
    def ship(self):     self._invalid("ship")
    def deliver(self):  self._invalid("deliver")
    def cancel(self):   self._invalid("cancel")

# --------------------------------
# Concrete states override only the σ they allow -> δ(q, σ) = q'

class PendingState(OrderState):              # q = Pending
    def confirm(self):
        # δ(Pending, confirm) = Confirmed
        ...
    def cancel(self):
        # δ(Pending, cancel) = Cancelled
        ...

class ConfirmedState(OrderState):            # q = Confirmed
    def ship(self):
        # δ(Confirmed, ship) = Shipped
        ...
    def cancel(self):
        # δ(Confirmed, cancel) = Cancelled
        ...

class ShippedState(OrderState):              # q = Shipped
    def deliver(self):
        # δ(Shipped, deliver) = Delivered
        ...

class DeliveredState(OrderState):            # q = Delivered (terminal; inherits all errors)
    ...

class CancelledState(OrderState):            # q = Cancelled (terminal; inherits all errors)
    ...

# --------------------------------
order = Order()          # q_0 = Pending
order.confirm()          # δ(Pending, confirm) = Confirmed
order.ship()             # δ(Confirmed, ship) = Shipped
order.deliver()          # δ(Shipped, deliver) = Delivered
try:
    order.cancel()       # δ(Delivered, cancel) undefined -> error
except InvalidTransitionError as e:
    print(f"Rejected: {e}")